# SWE015 – Gerçek zamanlı Türkçe chat moderasyonu: eğitim ve ölçümler
Google Colab'da **T4 GPU** ile çalıştırın. Hücreler sırayla: veri + normalizasyon, tf-idf, sözlük, word2vec + FFN, BERTurk, INT8 quantization, uygulama.

In [ ]:
# ===== HÜCRE 1: Kurulum, veri, normalizasyon =====
import os, re, json, time, unicodedata
import pandas as pd

# Modelleri ve sonuçları Google Drive'a kaydediyoruz (Colab kapanınca kaybolmasın)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/swe015'
except ImportError:
    BASE = './swe015'
os.makedirs(f'{BASE}/models', exist_ok=True)

# --- Veri: OffensEval-TR 2020 (Çöltekin, 2020), stefan-it'in train/dev/test bölmesi ---
from huggingface_hub import hf_hub_download

def load_split(name):
    path = hf_hub_download('stefan-it/offenseval2020_tr', f'{name}.txt', repo_type='dataset')
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip('\n')
            if not line.startswith('__label__'):
                continue
            label, text = line.split(' ', 1)
            rows.append({'text': text, 'label': int(label == '__label__OFF')})
    return pd.DataFrame(rows)

train, dev, test = load_split('train'), load_split('dev'), load_split('test')
for n, d in [('train', train), ('dev', dev), ('test', test)]:
    print(f'{n:5s}: {len(d):6d} mesaj, saldırgan oranı = {d.label.mean():.1%}')

# --- Normalizasyon (Alan B: regex) ---
TR_LOWER = str.maketrans({'I': 'ı', 'İ': 'i'})            # Türkçe'ye uygun küçük harf
TR_ASCII = str.maketrans('çğıöşü', 'cgiosu')               # "çok" ve "cok" aynı kelime olsun
LEET = {'0': 'o', '1': 'i', '3': 'e', '4': 'a', '5': 's', '7': 't', '@': 'a', '$': 's'}
LEET_RE = re.compile(r'(?<=[a-zçğıöşü])[013457@$]|[013457@$](?=[a-zçğıöşü])')

def normalize_light(text):
    """BERT için hafif temizlik: sadece etiketleri ve tekrarları düzelt."""
    text = re.sub(r'@USER', ' ', text)
    text = re.sub(r'\bURL\b', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)            # "çooook" -> "çook"
    return re.sub(r'\s+', ' ', text).strip()

def normalize_full(text):
    """Klasik modeller için tam normalizasyon."""
    text = normalize_light(text).translate(TR_LOWER).lower()
    for _ in range(2):                                        # "s1kt1r" gibi ardışık leet'ler için
        text = LEET_RE.sub(lambda m: LEET[m.group()], text)
    text = text.translate(TR_ASCII)
    text = re.sub(r'(.)\1+', r'\1', text)                     # tüm tekrarları teke indir
    text = re.sub(r'[^a-z\s]', ' ', text)                     # harf olmayanları at
    return re.sub(r'\s+', ' ', text).strip()

for df in (train, dev, test):
    df['clean'] = df.text.map(normalize_full)

print('\nÖrnek normalizasyon:')
for s in ['@USER Çoooook GÜZEL olmuş 2020 maçı!!! URL', 'b3n1 4rk4d4şım s4n4 n3 1st1y0r']:
    print(f'  {s!r}\n  -> {normalize_full(s)!r}')


In [ ]:
# ===== HÜCRE 2: Ölçüm yardımcıları + tf-idf baseline (Alan B) =====
import platform, joblib, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

RESULTS_PATH = f'{BASE}/results.json'
HARDWARE = f'{platform.processor() or platform.machine()} CPU, {os.cpu_count()} çekirdek (Colab)'

def measure_latency(predict_one, texts, n=200):
    """Tek tek mesajlarla ortalama gecikme (ms). Ödev en az 100 çalıştırma istiyor."""
    for t in texts[:10]:                      # ısınma turu, ölçüme dahil değil
        predict_one(t)
    t0 = time.perf_counter()
    for t in texts[:n]:
        predict_one(t)
    return (time.perf_counter() - t0) / n * 1000

def save_result(name, y_true, y_pred, size_mb, latency_ms):
    res = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
    res[name] = {'macro_f1': round(f1_score(y_true, y_pred, average='macro'), 4),
                 'size_mb': round(size_mb, 2), 'latency_ms': round(latency_ms, 2),
                 'hardware': HARDWARE}
    json.dump(res, open(RESULTS_PATH, 'w'), indent=2, ensure_ascii=False)
    print(classification_report(y_true, y_pred, target_names=['clean', 'offensive'], digits=4))
    print(pd.DataFrame(res).T)

# Kelime (1-2 gram) + karakter (2-5 gram) tf-idf, dengesiz sınıflar için class_weight
tfidf_model = make_pipeline(
    make_union(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
               TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=3, sublinear_tf=True)),
    LogisticRegression(C=4, class_weight='balanced', max_iter=2000))
tfidf_model.fit(train.clean, train.label)

path = f'{BASE}/models/tfidf_lr.joblib'
joblib.dump(tfidf_model, path)
pred = tfidf_model.predict(test.clean)
lat = measure_latency(lambda t: tfidf_model.predict([normalize_full(t)]), test.text.tolist())
save_result('1_tfidf_logreg', test.label, pred, os.path.getsize(path) / 1e6, lat)


In [ ]:
# ===== HÜCRE 3: Edit-distance sözlük eşleştirici, kural tabanlı baseline (Alan B) =====
import math, pickle
from collections import Counter

# 1) Sözlüğü veriden çıkar: saldırgan mesajlarda belirgin şekilde sık geçen kelimeler (log-odds oranı)
off_cnt, not_cnt = Counter(), Counter()
for text, y in zip(train.clean, train.label):
    (off_cnt if y else not_cnt).update(set(text.split()))
n_off, n_not = train.label.sum(), (1 - train.label).sum()
scores = {w: math.log((off_cnt[w] + 1) / (n_off + 2)) - math.log((not_cnt[w] + 1) / (n_not + 2))
          for w in off_cnt if off_cnt[w] >= 15 and len(w) >= 3}
LEXICON = [w for w in sorted(scores, key=scores.get, reverse=True) if scores[w] > 1.0][:150]
print(f'Sözlükte {len(LEXICON)} kelime var.')

# 2) Minimum edit distance (Levenshtein), dinamik programlama ile
def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def lexicon_hits(clean_text):
    """Sözlükteki bir kelimeye yeterince yakın olan kelimeleri döndürür (yazım hatalarını yakalar)."""
    hits = []
    for tok in clean_text.split():
        for w in LEXICON:
            max_d = 0 if len(w) <= 4 else 1           # kısa kelimelerde sadece birebir eşleşme
            if abs(len(tok) - len(w)) <= max_d and edit_distance(tok, w) <= max_d:
                hits.append(tok)
                break
    return hits

def lexicon_predict(text):
    return int(len(lexicon_hits(normalize_full(text))) > 0)

pred = [int(len(lexicon_hits(t)) > 0) for t in test.clean]
path = f'{BASE}/models/lexicon.pkl'
pickle.dump(LEXICON, open(path, 'wb'))
lat = measure_latency(lexicon_predict, test.text.tolist())
save_result('0_lexicon_editdist', test.label, pred, os.path.getsize(path) / 1e6, lat)
print('Örnek eşleşmeler:', edit_distance('aptal', 'aptl'), edit_distance('salak', 'salk'))


In [ ]:
# ===== HÜCRE 4: Kendi word2vec'imiz + feedforward ağ (Alan A) =====
import torch, torch.nn as nn, numpy as np, random
torch.manual_seed(0); np.random.seed(0); random.seed(0)
dev_ = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- 4a) Sözlük ---
tok_counts = Counter(w for t in train.clean for w in t.split())
vocab = ['<pad>', '<unk>'] + [w for w, c in tok_counts.items() if c >= 3]
w2i = {w: i for i, w in enumerate(vocab)}
def ids(text): return [w2i.get(w, 1) for w in text.split()]
print(f'Sözlük boyutu: {len(vocab)}')

# --- 4b) Skip-gram + negative sampling word2vec, sadece eğitim mesajlarıyla ---
EMB, WIN, NEG = 100, 3, 5
pairs = []
for t in train.clean:
    s = [i for i in ids(t) if i > 1]
    for i, c in enumerate(s):
        for j in range(max(0, i - WIN), min(len(s), i + WIN + 1)):
            if j != i: pairs.append((c, s[j]))
pairs = torch.tensor(pairs)
freq = torch.tensor([tok_counts.get(w, 0) for w in vocab], dtype=torch.float) ** 0.75
freq[:2] = 0; neg_dist = freq / freq.sum()
print(f'{len(pairs):,} (merkez, bağlam) çifti')

class SGNS(nn.Module):
    def __init__(s, V, D):
        super().__init__(); s.inp = nn.Embedding(V, D); s.out = nn.Embedding(V, D)
        nn.init.uniform_(s.inp.weight, -0.5 / D, 0.5 / D); nn.init.zeros_(s.out.weight)
    def forward(s, c, o, n):
        v = s.inp(c)
        pos = torch.log(torch.sigmoid((v * s.out(o)).sum(-1)))
        neg = torch.log(torch.sigmoid(-(s.out(n) @ v.unsqueeze(-1)).squeeze(-1))).sum(-1)
        return -(pos + neg).mean()

w2v = SGNS(len(vocab), EMB).to(dev_)
opt = torch.optim.Adam(w2v.parameters(), lr=0.003)
for ep in range(3):
    perm = torch.randperm(len(pairs)); tot = 0
    for k in range(0, len(pairs), 4096):
        b = pairs[perm[k:k + 4096]].to(dev_)
        n = torch.multinomial(neg_dist, len(b) * NEG, replacement=True).view(len(b), NEG).to(dev_)
        loss = w2v(b[:, 0], b[:, 1], n); opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
    print(f'word2vec epoch {ep + 1}: kayıp = {tot / (len(pairs) / 4096):.4f}')
E = w2v.inp.weight.detach().cpu()

def neighbors(w, k=5):
    if w not in w2i: return []
    sims = torch.nn.functional.cosine_similarity(E[w2i[w]].unsqueeze(0), E)
    return [vocab[i] for i in sims.topk(k + 1).indices.tolist()[1:]]
for w in ['mac', 'guzel', 'hakem']:
    print(f'  {w} -> {neighbors(w)}')

# --- 4c) Ortalama embedding -> feedforward ağ, SGD + backprop ile ---
class FFN(nn.Module):
    def __init__(s, E):
        super().__init__()
        s.emb = nn.EmbeddingBag.from_pretrained(E, freeze=False, mode='mean', padding_idx=0)
        s.net = nn.Sequential(nn.Linear(E.shape[1], 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 2))
    def forward(s, flat, offsets): return s.net(s.emb(flat, offsets))

def batchify(texts):
    seqs = [ids(t) or [1] for t in texts]
    offsets = torch.tensor([0] + [len(s) for s in seqs[:-1]]).cumsum(0)
    return torch.tensor([i for s in seqs for i in s]), offsets

ffn = FFN(E).to(dev_)
w = torch.tensor([1.0, (1 - train.label.mean()) / train.label.mean()], dtype=torch.float, device=dev_)   # sınıf dengesizliği
crit = nn.CrossEntropyLoss(weight=w)
opt = torch.optim.SGD(ffn.parameters(), lr=0.05, momentum=0.9)
X, Y = train.clean.tolist(), torch.tensor(train.label.values)

def predict_ffn(model, texts, device):
    model.eval()
    with torch.no_grad():
        f, o = batchify(texts)
        return model(f.to(device), o.to(device)).argmax(1).cpu().numpy()

best, best_state = 0, None
for ep in range(15):
    ffn.train(); perm = torch.randperm(len(X))
    for k in range(0, len(X), 64):
        idx = perm[k:k + 64]
        f, o = batchify([X[i] for i in idx])
        loss = crit(ffn(f.to(dev_), o.to(dev_)), Y[idx].to(dev_))
        opt.zero_grad(); loss.backward(); opt.step()            # backpropagation + SGD adımı
    f1 = f1_score(dev.label, predict_ffn(ffn, dev.clean.tolist(), dev_), average='macro')
    if f1 > best: best, best_state = f1, {k: v.clone() for k, v in ffn.state_dict().items()}
    print(f'FFN epoch {ep + 1:2d}: dev macro-F1 = {f1:.4f}')
ffn.load_state_dict(best_state)

ffn_cpu = ffn.cpu().eval()
path = f'{BASE}/models/w2v_ffn.pt'
torch.save({'state': ffn_cpu.state_dict(), 'vocab': vocab}, path)
pred = predict_ffn(ffn_cpu, test.clean.tolist(), 'cpu')
lat = measure_latency(lambda t: predict_ffn(ffn_cpu, [normalize_full(t)], 'cpu'), test.text.tolist())
save_result('2_word2vec_ffn', test.label, pred, os.path.getsize(path) / 1e6, lat)


In [ ]:
# ===== HÜCRE 5: BERTurk fine-tuning (Alan C) =====
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
MODEL_NAME = 'dbmdz/bert-base-turkish-cased'          # BERTurk
MAX_LEN, BS, EPOCHS, LR = 64, 32, 3, 2e-5

tok = AutoTokenizer.from_pretrained(MODEL_NAME)        # WordPiece alt-kelime tokenizer'ı
print('WordPiece örneği:', tok.tokenize('Bulaşık makinem su boşaltmıyor'))
bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(dev_)

def encode(texts):
    return tok([normalize_light(t) for t in texts], padding=True, truncation=True,
               max_length=MAX_LEN, return_tensors='pt')

def bert_predict(model, texts, device, bs=128):
    model.eval(); out = []
    with torch.no_grad():
        for k in range(0, len(texts), bs):
            enc = {a: b.to(device) for a, b in encode(texts[k:k + bs]).items()}
            out.append(model(**enc).logits.argmax(-1).cpu())
    return torch.cat(out).numpy()

opt = torch.optim.AdamW(bert.parameters(), lr=LR, weight_decay=0.01)
steps = EPOCHS * math.ceil(len(train) / BS)
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
cls_w = torch.tensor([1.0, (1 - train.label.mean()) / train.label.mean()], dtype=torch.float, device=dev_)
crit = nn.CrossEntropyLoss(weight=cls_w)                  # dengesiz sınıflar için ağırlık
scaler = torch.amp.GradScaler()
Xb, Yb = train.text.tolist(), torch.tensor(train.label.values)

BERT_DIR = f'{BASE}/models/berturk_offensive'
best = 0
for ep in range(EPOCHS):
    bert.train(); perm = torch.randperm(len(Xb)); t0 = time.time()
    for k in range(0, len(Xb), BS):
        idx = perm[k:k + BS]
        enc = {a: b.to(dev_) for a, b in encode([Xb[i] for i in idx]).items()}
        with torch.autocast('cuda', dtype=torch.float16):
            loss = crit(bert(**enc).logits.float(), Yb[idx].to(dev_))
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
    f1 = f1_score(dev.label, bert_predict(bert, dev.text.tolist(), dev_), average='macro')
    print(f'BERT epoch {ep + 1}: dev macro-F1 = {f1:.4f} ({time.time() - t0:.0f} sn)')
    if f1 > best:
        best = f1; bert.save_pretrained(BERT_DIR); tok.save_pretrained(BERT_DIR)

# En iyi modeli yükle; test metriği GPU'da, gecikme ise uygulamanın çalışacağı CPU'da ölçülür
bert = AutoModelForSequenceClassification.from_pretrained(BERT_DIR).to(dev_)
pred = bert_predict(bert, test.text.tolist(), dev_)
bert_cpu = AutoModelForSequenceClassification.from_pretrained(BERT_DIR).eval()
lat = measure_latency(lambda t: bert_predict(bert_cpu, [t], 'cpu'), test.text.tolist(), n=100)
size = sum(os.path.getsize(f'{BERT_DIR}/{f}') for f in os.listdir(BERT_DIR) if f.endswith(('.safetensors', '.bin')))
save_result('3_berturk_fp32', test.label, pred, size / 1e6, lat)


In [ ]:
# ===== HÜCRE 6: BERTurk'ü INT8'e quantize etme (Alan D) =====
# Dinamik quantization: Linear katmanların ağırlıkları INT8'e çevrilir, aktivasyonlar çalışırken quantize edilir.
LATENCY_BUDGET_MS = 50                                    # kendi koyduğumuz hedef: CPU'da mesaj başına 50 ms
qbert = torch.ao.quantization.quantize_dynamic(bert_cpu, {nn.Linear}, dtype=torch.qint8).eval()

qpath = f'{BASE}/models/berturk_int8.pt'
torch.save(qbert.state_dict(), qpath)
pred = bert_predict(qbert, test.text.tolist(), 'cpu', bs=64)     # quantized model sadece CPU'da çalışır
lat = measure_latency(lambda t: bert_predict(qbert, [t], 'cpu'), test.text.tolist(), n=100)
save_result('4_berturk_int8', test.label, pred, os.path.getsize(qpath) / 1e6, lat)
print(f'\nGecikme hedefi {LATENCY_BUDGET_MS} ms -> INT8: {lat:.1f} ms '
      f'({"tuttu" if lat <= LATENCY_BUDGET_MS else "tutmadı"})')


In [ ]:
# ===== HÜCRE 7: Uygulamayı Colab'da başlat (repo kodu ile) =====
# Repo'yu Colab'a klonlayıp eğitilen modellerle başlatır; herkese açık bir link verir.
REPO_URL = 'https://github.com/azizealtinel/swe015-chat-moderation.git'
!git clone -q {REPO_URL} /content/app_repo
import sys; sys.path.insert(0, '/content/app_repo')
from src.models import load_all
from app import build_ui
build_ui(load_all(f'{BASE}/models', include_fp32=True)).launch(share=True)
